In [1]:
!killall -9 python
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

/bin/bash: line 1: killall: command not found
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:3 http://archive.ubuntu.com/ubuntu jammy-backports InRelease           
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
28 packages can be upgraded. Run 'apt list --upgradable' to see them.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pciutils is already the newest version (1:3.7.0-6).
0 upgraded, 0 newly installed, 0 to remove and 28 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to r

In [1]:
import os
from huggingface_hub import login

# You can get the token from an environment variable for security

# Or paste it directly (less secure)
hf_token = os.environ.get("HF_TOKEN")

login(token=hf_token)

In [2]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

Error: listen tcp 127.0.0.1:11434: bind: address already in use


In [17]:
!pip install -r requirements.txt

Processing /home/conda/feedstock_root/build_artifacts/bld/rattler-build_alembic_1770806831/work (from -r requirements.txt (line 7))
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/home/conda/feedstock_root/build_artifacts/bld/rattler-build_alembic_1770806831/work'



In [ ]:
!ollama serve

# Mistral Models

In [3]:
!ollama pull mistral

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling f5074b1221da: 100% ▕██████████████████▏ 4.4 GB                         
pulling 43070e2d4e53: 100% ▕██████████████████▏  11 KB                         
pulling 1ff5b64b61b9: 100% ▕██████████████████▏  799 B                         
pulling ed11eda7790d: 100% ▕██████████████████▏   30 B                         
pulling 1064e17101bd: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 


In [8]:
import ollama


response = ollama.chat(
    model='mistral',
    messages=[{'role': 'user', 'content': 'Hello!'}],
)
print(response.message.content)

 Hello! How can I assist you today?

* If you're looking for a chat about a specific topic, feel free to ask.
* If you're just here to chat, I'm happy to talk about a variety of subjects.
* If you have any questions, I'll do my best to help you find the answers you need.

Let me know how I can be of service! 😊


In [9]:
import pandas as pd
import numpy as np

#Uploading the data and transfering it to a dataframe.
my_text = pd.read_csv("Test Corpus.txt", sep=";")

my_text.head()
my_text['English'] = my_text.iloc[:, 1].str.strip()
my_text['Spanish'] = my_text['Spanish'].str.strip()
my_english_text = my_text.English.to_list()
my_spanish_text = my_text.Spanish.to_list()


In [5]:
my_base_translations = []

for text in my_spanish_text:
    response = ollama.chat(
        model='mistral',
        messages=[
            # System role sets the behavior
            {'role': 'system', 'content': 'You are a professional translator. Translate the Spanish text to English. Provide only the translation without quotes or explanations.'},
            {'role': 'user', 'content': text}
        ],
        options={'temperature': 0}
    )

    #Print the results
    print(f"Original: {text}")
    print(f"English:  {response.message.content}\n")
    my_base_translations.append(response.message.content)

Original: Buenos días.
English:   Good morning.

Original: Buenas tardes.
English:   Good afternoon.

Original: Buenas noches.
English:   Good night.

Original: Hola.
English:   Hello.

Original: ¿Cómo estás?
English:   How are you?

Original: Bien, gracias. ¿Y tú?
English:   Sure, thank you. And you?

Original: ¿Qué pasa?
English:   What's up?

Original: ¿Qué tal?
English:   How are you?

Original: ¿Qué haces?
English:   What are you doing?

Original: ¿Cómo te va?
English:   How are you?

Original: Muy bien.
English:   Very well.

Original: Todo bien.
English:   Everything's fine.

Original: Más o menos.
English:   Approximately.

Original: Así, así.
English:   Here, here.

Original: No muy bien.
English:   Not good.

Original: ¡Adiós!
English:   Goodbye!

Original: Hasta mañana.
English:   See you tomorrow.

Original: Hasta luego.
English:   See you later.

Original: Hasta pronto.
English:   See you soon.

Original: Nos vemos.
English:   See you.

Original: Gracias.
English:   Thank 

In [6]:
import sacrebleu
my_base_translations = [s.strip() for s in my_base_translations]

bleu = sacrebleu.corpus_bleu(my_base_translations, my_english_text)
chrf = sacrebleu.corpus_chrf(my_base_translations, my_english_text)

print(f"BLEU Score: {bleu.score:.2f}")
print(f"chrF Score: {chrf.score:.2f}")

BLEU Score: 3.67
chrF Score: 30.93


In [7]:
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "info"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
import lightning.pytorch as pl
import pytorch_lightning
from comet import download_model, load_from_checkpoint

# 1. The Bridge Fix
# Ensure the legacy mapping exists for COMET's internal calls
if not hasattr(pytorch_lightning, 'utilities'):
    pytorch_lightning.utilities = pl.utilities

# 2. Model Loading
# It is often safer to download and load in one go if you aren't managing paths manually
model_path = download_model("Unbabel/wmt22-comet-da")
model = load_from_checkpoint(model_path)

# 3. Data Preparation
# Ensure your input lists (my_spanish_text, etc.) are already defined
data = [
    {"src": src, "mt": mt, "ref": ref}
    for src, mt, ref in zip(my_spanish_text, my_base_translations, my_english_text)
]

# 4. Predict Logic
# Note: Newer versions of COMET use 'gpus' as an integer or list.
# If you are on a CPU-only machine, set gpus=0.
model_output = model.predict(
    data, 
    batch_size=32, 
    gpus=1 if torch.cuda.is_available() else 0
)

# 5. Output
print(f"System Score: {model_output.system_score:.4f}")

I0000 00:00:1777398997.747206     431 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
loading file sentencepiece.bpe.model from cache at /home/jovyan/.cache/huggingface/hub/models--xlm-roberta-large/snapshots/c23d21b0620b635a76227c604d44e43a9f0ee389/sentencepiece.bpe.model
loading file tokenizer.json from cache at /home/jovyan/.cache/huggingface/hub/models--xlm-roberta-large/snapshots/c23d21b0620b635a76227c604d44e43a9f0ee389/tokenizer.json
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at /home/jovyan/.cache/huggingface/hub/models--xlm-roberta-large/snapshots/c23d21b0620b635a76227c604d44e43a9f0ee389/tokenizer_config.json
loading file chat_tem

System Score: 0.8961


In [4]:
#Implement QLoRA from Unsloth
import os
os.environ["TORCH_COMPILE_DISABLE"] = "1"

from unsloth import FastLanguageModel, unsloth_train
import torch
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorWithPadding
from datasets import load_dataset

torch.cuda.empty_cache()

# --- 1. Load Model & Tokenizer ---
max_seq_length = 512
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-bnb-4bit", # Pre-quantized for speed
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# --- 2. Add QLoRA Adapters ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank: higher = more "memory" but more VRAM
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0.1, # Optimized at 0
    bias = "none",    # Optimized at none
    use_gradient_checkpointing = "unsloth",
)

# --- 3. Prepare Dataset ---
prompt_style = "Spanish: {}\nEnglish: {}"

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Mistral/Llama standard for training
tokenizer.model_max_length = max_seq_length 

def formatting_prompts_func(examples):
    translations = examples["translation"]
    texts = []
    for entry in translations:
        # Extract en and es from the nested dictionary
        en_text = entry["en"]
        es_text = entry["es"]
        # Format and append EOS_TOKEN
        text = f"Spanish: {es_text}\nEnglish: {en_text}{tokenizer.eos_token}"
        texts.append(text)
    return { "text" : texts }
    
def tokenize_function(examples):
    # Format the text
    texts = [f"Spanish: {es}\nEnglish: {en}{tokenizer.eos_token}" 
             for es, en in zip([x["es"] for x in examples["translation"]], 
                               [x["en"] for x in examples["translation"]])]
    
    # Tokenize
    tokenized = tokenizer(
        texts,
        truncation = True,
        max_length = max_seq_length,
        padding = "max_length",
    )
    
    # FOR CAUSAL LM: Labels must be a copy of input_ids
    tokenized["labels"] = [list(ids) for ids in tokenized["input_ids"]]
    return tokenized
# Load a subset to avoid OOM or long wait times
dataset = load_dataset("Helsinki-NLP/europarl", "en-es", split="train[:50000]")
dataset = dataset.map(
    tokenize_function,
    batched = True,
    remove_columns = dataset.column_names,
    desc = "Manually tokenizing and aligning labels"
)

# --- 4. Start Training ---
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    #ataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer),
    packing = True,
    args = TrainingArguments(
        per_device_train_batch_size = 1, # You can likely increase this with Unsloth
        gradient_accumulation_steps = 16,
        warmup_steps = 5,
        #max_steps = 60,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        group_by_length = False,
        report_to = 'none', 
        average_tokens_across_devices=False,
        remove_unused_columns = False,
    ),
)
unsloth_train(trainer)

# --- 5. After trainer.train(), run Inference ---
FastLanguageModel.for_inference(model) # Enable 2x faster inference


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/jovyan/.conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1773494585.099322 2181103 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
comet_ml is installed but the Comet API Key is not configured. Please set the `COMET_API_KEY` environment variable to enable Comet logging. Check out the documentation for other ways of configuring it: https://www.comet.com/docs/v2/guides/experiment-management/configure-sdk/#set-the-api-key


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    Tesla V100-PCIE-32GB. Num GPUs = 4. Max memory: 31.739 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu126. CUDA: 7.0. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.4 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50,000 | Num Epochs = 1 | Total steps = 782
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 16 x 1) = 64
 "-____-"     Trainable parameters = 41,943,040 of 7,283,675,136 (0.58% trained)


Step,Training Loss
1,54.946300
2,63.280300
3,62.770900
4,70.539200
5,62.122400
6,25.909700
7,6.394000
8,2.456800
9,2.042600
10,1.950600


comet_ml is installed but the Comet API Key is not configured. Please set the `COMET_API_KEY` environment variable to enable Comet logging. Check out the documentation for other ways of configuring it: https://www.comet.com/docs/v2/guides/experiment-management/configure-sdk/#set-the-api-key
comet_ml is installed but the Comet API Key is not configured. Please set the `COMET_API_KEY` environment variable to enable Comet logging. Check out the documentation for other ways of configuring it: https://www.comet.com/docs/v2/guides/experiment-management/configure-sdk/#set-the-api-key


<s> Spanish: Hola, ¿cómo estás hoy?
English: Hello, how are you today?</s>


In [5]:
model.save_pretrained_gguf("QLoRAmodel", tokenizer, quantization_method = "q4_k_m")

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /home/jovyan/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00003.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 3/3 [00:59<00:00, 19.86s/it]


Unsloth: Merge process complete. Saved to `/home/jovyan/work/Ollama Models/QLoRAmodel`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Found existing llama.cpp at `/home/jovyan/work/Ollama Models/llama.cpp` -- using it.
Unsloth: Note: the default location has moved to `/home/jovyan/.unsloth/llama.cpp`.
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['QLoRAmodel_gguf/mistral-7b.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['QLoRAmodel_gguf/mistral-7b.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/mistral-7b'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /home/jovyan/.unsloth/llama.cpp/llama-cli --model QLoRAmodel_gguf/mistral-7b.Q4_K_M.gguf -p "why is the sky blue?"


{'save_directory': 'QLoRAmodel',
 'gguf_directory': 'QLoRAmodel_gguf',
 'gguf_files': ['QLoRAmodel_gguf/mistral-7b.Q4_K_M.gguf'],
 'modelfile_location': None,
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [20]:
!find . -name "Modelfile"

./QLoRAmodel/Modelfile


In [6]:
!ollama create qloratranslator -f "/home/jovyan/work/Ollama Models/QLoRAmodel_gguf/Modelfile"

]11;?\gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠴ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠋ gathering model components ⠋ gathering model components ⠙ gathering model components ⠸ gathering model components ⠸ gathering model components ⠼ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠏ gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠴ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠏ gathering model components ⠙ gathering model components ⠙ gathering model components ⠹ gathering model components ⠼ gather

In [43]:
my_QLoRA_translations = []

for text in my_spanish_text:
    prompt = f"Spanish: {text}\nEnglish:"
    response = ollama.generate(
        model='qloratranslator',      
        prompt=prompt,
        options={'temperature': 0,
                'stop': ["Spanish:", "\n", "English:"]}
    )
    translation = response['response'].strip()

    #Print the results
    print(f"Original: {text}")
    print(f"English:  {translation}\n")
    my_QLoRA_translations.append(translation)



HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


Original: Buenos días.
English:  It is a



HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


Original: Buenas tardes.
English:  I would like to thank the rapporteur, Mr



HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


...

Original: ¿Tiene…?
English:  Mr President, the



HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


Original: Me gustaría…
English:  I would like to see...



HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


Original: Quisiera…
English:  I would like to see...



In [44]:
import sacrebleu
bleu = sacrebleu.corpus_bleu(my_QLoRA_translations, my_english_text)
chrf = sacrebleu.corpus_chrf(my_QLoRA_translations, my_english_text)

print(f"BLEU Score: {bleu.score:.2f}")
print(f"chrF Score: {chrf.score:.2f}")

BLEU Score: 1.17
chrF Score: 18.87


In [45]:
# 1. Setup your data
# 'src' = original Spanish, 'mt' = your Ollama output, 'ref' = ground truth
data = [
    {
        "src": src,
        "mt": mt,
        "ref": ref
    }
    for src, mt, ref in zip(my_spanish_text, my_QLoRA_translations, my_english_text)
]


# 3. Calculate scores
model_output = model.predict(data, batch_size=8, gpus=1) # Set gpus=1 if you have a GPU

# 4. Access the results
print(f"System Score: {model_output.system_score}") 
# Individual scores are in model_output.scores

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]
/home/jovyan/.conda/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 14/14 [00:01<00:00,  9.93it/s]


System Score: 0.6682110318431148


# Llama2 Models:

In [ ]:
!ollama pull llama2:7b

In [20]:
my_base_translations = []

for text in my_spanish_text:
    response = ollama.chat(
        model='llama2:7b',
        messages=[
            # System role sets the behavior
            {'role': 'system', 'content': 'You are a professional translator. Translate the Spanish text to English. Provide only the translation without quotes or explanations.'},
            {'role': 'user', 'content': text}
        ],
        options={'temperature': 0}
    )

    #Print the results
    print(f"Original: {text}")
    print(f"English:  {response.message.content}\n")
    my_base_translations.append(response.message.content)

Original: Buenos días.
English:  Good morning.

Original: Buenas tardes.
English:  Bright and early!

Original: Buenas noches.
English:  Good evening.

Original: Hola.
English:  Hello

Original: ¿Cómo estás?
English:  How are you?

Original: Bien, gracias. ¿Y tú?
English:  How are you?

Original: ¿Qué pasa?
English:  What's up?

Original: ¿Qué tal?
English:  How are you?

Original: ¿Qué haces?
English:  What are you doing?

Original: ¿Cómo te va?
English:  How are you?

Original: Muy bien.
English:  Very well.

Original: Todo bien.
English:  Everything is fine.

Original: Más o menos.
English:  More or less.

Original: Así, así.
English:  Okay. Here is the translation of "Así, así" from Spanish to English:

So

Original: No muy bien.
English:  Not very well.

Original: ¡Adiós!
English:  Goodbye!

Original: Hasta mañana.
English:  See you tomorrow.

Original: Hasta luego.
English:  Bye

Original: Hasta pronto.
English:  See you soon.

Original: Nos vemos.
English:  See you later.

Origi

In [14]:
my_base_translations = [s.strip() for s in my_base_translations]

bleu = sacrebleu.corpus_bleu(my_base_translations, my_english_text)
chrf = sacrebleu.corpus_chrf(my_base_translations, my_english_text)

print(f"BLEU Score: {bleu.score:.2f}")
print(f"chrF Score: {chrf.score:.2f}")

BLEU Score: 4.03
chrF Score: 34.09


In [15]:
# 1. Setup your data
# 'src' = original Spanish, 'mt' = your Ollama output, 'ref' = ground truth
data = [
    {
        "src": src,
        "mt": mt,
        "ref": ref
    }
    for src, mt, ref in zip(my_spanish_text, my_base_translations, my_english_text)
]

print(f"CUDA Available: {torch.cuda.is_available()}")

# 2. Load the pre-trained model
# 'wmt22-comet-da' is a standard high-performing version

# 3. Calculate scores
model_output = model.predict(data, batch_size=8, gpus=1) # Set gpus=1 if you have a GPU

# 4. Access the results
print(f"System Score: {model_output.system_score}") 
# Individual scores are in model_output.scores

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


CUDA Available: True


/home/jovyan/.conda/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 14/14 [00:01<00:00, 10.72it/s]


System Score: 0.9042720709134031


In [6]:
import os
os.environ["TORCH_COMPILE_DISABLE"] = "1"

from unsloth import FastLanguageModel, unsloth_train
import torch
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorWithPadding
from datasets import load_dataset


torch.cuda.empty_cache()

# --- 1. Load Model & Tokenizer ---
max_seq_length = 512
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-2-7b-chat-bnb-4bit", # Pre-quantized for speed
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# --- 2. Add QLoRA Adapters ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank: higher = more "memory" but more VRAM
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0.1, # Optimized at 0
    bias = "none",    # Optimized at none
    use_gradient_checkpointing = "unsloth",
)

# --- 3. Prepare Dataset ---
prompt_style = "Spanish: {}\nEnglish: {}"

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Mistral/Llama standard for training

def formatting_prompts_func(examples):
    translations = examples["translation"]
    texts = []
    for entry in translations:
        # Extract en and es from the nested dictionary
        en_text = entry["en"]
        es_text = entry["es"]
        # Format and append EOS_TOKEN
        text = f"Spanish: {es_text}\nEnglish: {en_text}{tokenizer.eos_token}"
        texts.append(text)
    return { "text" : texts }


def tokenize_function(examples):
    # Format the text
    texts = [f"Spanish: {es}\nEnglish: {en}{tokenizer.eos_token}" 
             for es, en in zip([x["es"] for x in examples["translation"]], 
                               [x["en"] for x in examples["translation"]])]
    
    # Tokenize
    tokenized = tokenizer(
        texts,
        truncation = True,
        max_length = max_seq_length,
        padding = "max_length",
    )
    
    # FOR CAUSAL LM: Labels must be a copy of input_ids
    tokenized["labels"] = [list(ids) for ids in tokenized["input_ids"]]
    return tokenized
# Load a subset to avoid OOM or long wait times
dataset = load_dataset("Helsinki-NLP/europarl", "en-es", split="train[:50000]")
dataset = dataset.map(
    tokenize_function,
    batched = True,
    remove_columns = dataset.column_names,
    desc = "Manually tokenizing and aligning labels"
)

# --- 4. Start Training ---
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    #ataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer),
    packing = True,
    args = TrainingArguments(
        per_device_train_batch_size = 1, # You can likely increase this with Unsloth
        gradient_accumulation_steps = 16,
        warmup_steps = 5,
        #max_steps = 60,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        group_by_length = False,
        report_to = 'none', 
        average_tokens_across_devices=False,
        remove_unused_columns = False,
    ),
)
unsloth_train(trainer)

# --- 5. After trainer.train(), run Inference ---
FastLanguageModel.for_inference(model) # Enable 2x faster inference

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla V100-PCIE-32GB. Num GPUs = 4. Max memory: 31.739 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu126. CUDA: 7.0. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50,000 | Num Epochs = 1 | Total steps = 782
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 16 x 1) = 64
 "-____-"     Trainable parameters = 39,976,960 of 6,778,392,576 (0.59% trained)


Step,Training Loss
1,35.463300
2,40.369700
3,35.470700
4,24.948400
5,9.636100
6,5.766400
7,4.119400
8,3.344600
9,2.887300
10,2.788700


comet_ml is installed but the Comet API Key is not configured. Please set the `COMET_API_KEY` environment variable to enable Comet logging. Check out the documentation for other ways of configuring it: https://www.comet.com/docs/v2/guides/experiment-management/configure-sdk/#set-the-api-key
comet_ml is installed but the Comet API Key is not configured. Please set the `COMET_API_KEY` environment variable to enable Comet logging. Check out the documentation for other ways of configuring it: https://www.comet.com/docs/v2/guides/experiment-management/configure-sdk/#set-the-api-key


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              

In [7]:
model.save_pretrained_gguf("QLoRALLamamodel", tokenizer, quantization_method = "q4_k_m")

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /home/jovyan/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00003.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 3/3 [00:56<00:00, 18.82s/it]


Unsloth: Merge process complete. Saved to `/home/jovyan/work/Ollama Models/QLoRALLamamodel`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Found existing llama.cpp at `/home/jovyan/work/Ollama Models/llama.cpp` -- using it.
Unsloth: Note: the default location has moved to `/home/jovyan/.unsloth/llama.cpp`.
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['QLoRALLamamodel_gguf/llama-2-7b-chat.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: All GGUF conversions completed successfully!
Generated files: ['QLoRALLamamodel_gguf/llama-2-7b-chat.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /home/jovyan/.unsloth/llama.cpp/llama-cli --model QLoRALLamamodel_gguf/llama-2-7b-chat.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to QLoRALLamamodel_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f QLoRALLamamodel_gguf/Modelfile


{'save_directory': 'QLoRALLamamodel',
 'gguf_directory': 'QLoRALLamamodel_gguf',
 'gguf_files': ['QLoRALLamamodel_gguf/llama-2-7b-chat.Q4_K_M.gguf'],
 'modelfile_location': 'QLoRALLamamodel_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': True}

In [10]:
!ollama create qlorallamatranslator -f "/home/jovyan/work/Ollama Models/QLoRALLamamodel_gguf/Modelfile"

]11;?\gathering model components ⠙ gathering model components ⠹ gathering model components ⠹ gathering model components ⠼ gathering model components ⠼ gathering model components ⠴ gathering model components ⠧ gathering model components ⠧ gathering model components ⠏ gathering model components ⠏ gathering model components ⠙ gathering model components ⠹ gathering model components ⠹ gathering model components ⠼ gathering model components ⠼ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠋ gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠼ gathering model components ⠼ gathering model components ⠦ gathering model components ⠦ gathering model components ⠇ gathering model components ⠇ gathering model components ⠋ gathering model components ⠋ gathering model components ⠙ gathering model components ⠸ gathering model components ⠼ gather

In [46]:
my_QLoRA_translations = []

for text in my_spanish_text:
    prompt = f"Spanish: {text}\nEnglish:"
    response = ollama.generate(
        model='qlorallamatranslator',      
        prompt=prompt,
        options={'temperature': 0,
                'stop': ["Spanish:", "\n", "English:"]}
    )
    translation = response['response'].strip()

    #Print the results
    print(f"Original: {text}")
    print(f"English:  {translation}\n")
    my_QLoRA_translations.append(translation)


HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


Original: Buenos días.
English:  I wish you a good day.



HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


Original: Buenas tardes.
English:  Good evening.



HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


...

Original: ¿Tiene…?
English:  Does it have a view on this?



HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


Original: Me gustaría…
English:  I would like to see this.



HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


Original: Quisiera…
English:  I would like to thank the Commission for its work.



In [47]:
my_QLoRA_translations = [s.strip("English: ") for s in my_QLoRA_translations]

bleu = sacrebleu.corpus_bleu(my_QLoRA_translations, my_english_text)
chrf = sacrebleu.corpus_chrf(my_QLoRA_translations, my_english_text)

print(f"BLEU Score: {bleu.score:.2f}")
print(f"chrF Score: {chrf.score:.2f}")

BLEU Score: 2.39
chrF Score: 27.78


In [48]:
data = [
    {
        "src": src,
        "mt": mt,
        "ref": ref
    }
    for src, mt, ref in zip(my_spanish_text, my_QLoRA_translations, my_english_text)
]


# 3. Calculate scores
model_output = model.predict(data, batch_size=8, gpus=1) # Set gpus=1 if you have a GPU

# 4. Access the results
print(f"System Score: {model_output.system_score}") 
# Individual scores are in model_output.scores

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]
/home/jovyan/.conda/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 14/14 [00:01<00:00,  9.74it/s]


System Score: 0.7694843750860956


In [22]:
print(my_QLoRA_translations)

['Good morning.', 'Good evening.', 'Goodnight.', 'Hello.', 'How are you?', 'Thank you.', 'What is happening?', '', 'What is your job?', 'How is it going?', '', 'All right, then.', '', '', '', 'Goodbye!', 'Until tomorrow.', 'Goodbye.', 'Goodbye.', 'We see each other.', 'Thank you.', 'Please.', 'This is not a matter of any importance.', '', '', '', '', 'Good afternoon, Mr President.', 'Good luck!', 'Good luck!', 'You are a professional translator, so you will translate this English text into English.', 'You are a professional translator.', 'What is your name?', 'I am Michael.', 'Thank you very much.', '', 'I am delighted.', 'I am a professional translator.', 'You are a professional translator.', 'I am from the United States.', 'What age do you have?', 'How old are you?', 'When is your birthday?', 'What do you do?', 'I am an astronaut.', 'What do you do?', 'I work for NASA', 'What would you like to do?', 'I like reading it.', 'I love eating.', 'What do you do in your free time?', 'What do

## FAISS Implementation

In [6]:
import pandas as pd
#Cleaning the RAG document.
ragdoc = pd.read_csv("spa.txt", sep='\t')

ragdoc = ragdoc.iloc[:, :-1]

ragdoc = ragdoc.iloc[:, ::-1]

ragdoc.to_csv('ragdoc.txt', sep='\t', index=False)



In [9]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

# Load the document
loader = TextLoader("ragdoc.txt")
documents = loader.load()

# Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=30)
docs = text_splitter.split_documents(documents=documents)

TensorFlow version 2.21.0 available.


In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Load embedding model
embedding_model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {"device": "cuda"}
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs=model_kwargs
)

# Create FAISS vector store
vectorstore = FAISS.from_documents(docs, embeddings)

# Save and reload the vector store
vectorstore.save_local("faiss_index_")
persisted_vectorstore = FAISS.load_local("faiss_index_", embeddings, allow_dangerous_deserialization=True)

# Create a retriever
retriever = persisted_vectorstore.as_retriever()

Load pretrained SentenceTransformer: sentence-transformers/all-mpnet-base-v2
loading configuration file config.json from cache at /home/jovyan/.cache/huggingface/hub/models--sentence-transformers--all-mpnet-base-v2/snapshots/e8c3b32edf5434bc2275fc9bab85f82640a19130/config.json
Model config MPNetConfig {
  "architectures": [
    "MPNetForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "mpnet",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "relative_attention_num_buckets": 32,
  "transformers_version": "4.57.6",
  "vocab_size": 30527
}

loading weights file model.safetensors from cache at /home/jovyan/.cache/huggingface/hub/models--sentence-transformers--all-mpnet-base-v2/snapshots/e8c3b32edf5434bc227

In [11]:
# Assuming 'retriever' and 'my_spanish_text' are already defined
my_rag_translations = []

for text in my_spanish_text:
    # 1. Search the FAISS index for relevant context
    # This finds the top chunks in your PDF related to the current Spanish snippet
    docs = retriever.invoke(text)
    context = "\n".join([doc.page_content for doc in docs])

    # 2. Build the augmented prompt
    system_prompt = (
        "You are a professional translator. Translate the Spanish text to English. "
        "Use the following provided context for terminology and style guidance if relevant. "
        "Provide only the translation without quotes or explanations.\n\n"
        f"### CONTEXT FROM DOCUMENTS:\n{context}"
    )

    # 3. Call Ollama with the context-aware prompt
    response = ollama.chat(
        model='mistral',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': f"Translate: {text}"}
        ],
        options={'temperature': 0}
    )

    translation = response.message.content.strip()
    
    print(f"Original: {text}")
    print(f"English:  {translation}\n")
    my_rag_translations.append(translation)

HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Original: Buenos días.
English:  Good morning.



HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Original: Buenas tardes.
English:  Good afternoon.



HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


...

Original: ¿Tiene…?
English:  Do you have...?
¿Tenéis…?
¿Conoces a Tom...?
¿La conoces a ella...?
¿Vos la conocés...?
¿La conoces...?
¿La conocéis...?
¿La conoce...?
¿La conocen...?
¿Vosotros la conocéis...?
¿Le conoces...?
¿Le conocéis...?
¿Le conoce...?
¿Ustedes le conocen...?
¿Sabés por qué...?
¿Te gusta Tom...?
¿Te gusta el rap...?
¿Te gusta el té...?
¿Le gusta el té...?
¿Os gusta el té...?
¿Les gusta el té...?
¿Quieres a Tom...?
¿Echas de menos a Tom...?
¿Recuerdas...?
¿Lo recuerdas...?
¿Tú crees...?
¿Usted cree...?
¿Confiás en mí...?
¿Confías en mí...?
¿Confiáis en mí...?
¿Me quedo...?
Cierra la puerta.
Cierra la puerta.
Siéntate junto a mí.
Sentate Tomás.
Algunos peces vuelan.
Alguien llegó.
Habla claramente.
Hable claramente.
Hablan claramente.
Hablad claramente.
Habla quedito.
Habla calladito.
Párense en línea.
Empieza a correr.
Comienza a correr.
Empieza a cantar.
Empiece a cantar.
Empezad a cantar.
Empiecen a cantar.
Comienza a cantar.
Comience a cantar.
Comenzad a cantar.
Co

HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Original: Me gustaría…
English:  I would like...



HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Original: Quisiera…
English:  I want...



In [12]:
bleu = sacrebleu.corpus_bleu(my_rag_translations, my_english_text)
chrf = sacrebleu.corpus_chrf(my_rag_translations, my_english_text)

print(f"BLEU Score: {bleu.score:.2f}")
print(f"chrF Score: {chrf.score:.2f}")

BLEU Score: 3.67
chrF Score: 30.00


In [13]:
# 1. Setup your data
# 'src' = original Spanish, 'mt' = your Ollama output, 'ref' = ground truth
data = [
    {
        "src": src,
        "mt": mt,
        "ref": ref
    }
    for src, mt, ref in zip(my_spanish_text, my_rag_translations, my_english_text)
]

print(f"CUDA Available: {torch.cuda.is_available()}")

# 2. Load the pre-trained model
# 'wmt22-comet-da' is a standard high-performing version

# 3. Calculate scores
model_output = model.predict(data, batch_size=8, gpus=1) # Set gpus=1 if you have a GPU

# 4. Access the results
print(f"System Score: {model_output.system_score}") 
# Individual scores are in model_output.scores

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


CUDA Available: True


/home/jovyan/.conda/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 14/14 [00:01<00:00,  7.84it/s]


System Score: 0.9031851040544333


In [37]:
my_QLoRA_RAG_translations = []

for text in my_spanish_text:
    # 1. Retrieve context relative to the Spanish snippet
    docs = retriever.invoke(text)
    context = "\n".join([doc.page_content for doc in docs])

    # 2. Integrate context into your specific QLoRA prompt template
    # We place the context BEFORE the translation task so the model "reads" it first
    prompt = (
        f"Context: {context}\n"
        f"Spanish: {text}\n"
        "English:"
    )

    # 3. Generate using your fine-tuned model
    response = ollama.generate(
        model='qloratranslator',      
        prompt=prompt,
        options={
            'temperature': 0,
            'stop': ["Spanish:", "\n", "English:", "Context:"] # Added "Context:" as a stop sequence
        }
    )
    
    translation = response['response'].strip()

    print(f"Original: {text}")
    print(f"English:  {translation}\n")
    my_QLoRA_RAG_translations.append(translation)

HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


Original: Buenos días.
English:  Thank you.



HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


Original: Buenas tardes.
English:  It is all



HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


...

Original: ¿Tiene…?
English:  He had to



HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


Original: Me gustaría…
English:  I would like to see...



HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


Original: Quisiera…
English:  I would like to see...



In [38]:
bleu = sacrebleu.corpus_bleu(my_QLoRA_RAG_translations, my_english_text)
chrf = sacrebleu.corpus_chrf(my_QLoRA_RAG_translations, my_english_text)

print(f"BLEU Score: {bleu.score:.2f}")
print(f"chrF Score: {chrf.score:.2f}")

BLEU Score: 2.24
chrF Score: 28.30


In [39]:
# 1. Setup your data
# 'src' = original Spanish, 'mt' = your Ollama output, 'ref' = ground truth
data = [
    {
        "src": src,
        "mt": mt,
        "ref": ref
    }
    for src, mt, ref in zip(my_spanish_text, my_QLoRA_RAG_translations, my_english_text)
]

print(f"CUDA Available: {torch.cuda.is_available()}")

# 2. Load the pre-trained model
# 'wmt22-comet-da' is a standard high-performing version

# 3. Calculate scores
model_output = model.predict(data, batch_size=8, gpus=1) # Set gpus=1 if you have a GPU

# 4. Access the results
print(f"System Score: {model_output.system_score}") 
# Individual scores are in model_output.scores

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


CUDA Available: True


/home/jovyan/.conda/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 14/14 [00:01<00:00,  9.88it/s]


System Score: 0.8011606984116413


In [21]:
# Assuming 'retriever' and 'my_spanish_text' are already defined
my_rag_translations = []

for text in my_spanish_text:
    # 1. Search the FAISS index for relevant context
    # This finds the top chunks in your PDF related to the current Spanish snippet
    docs = retriever.invoke(text)
    context = "\n".join([doc.page_content for doc in docs])

    # 2. Build the augmented prompt
    system_prompt = (
        "You are a professional translator. Translate the Spanish text to English. "
        "Use the following provided context for terminology and style guidance if relevant. "
        "Provide only the translation without quotes or explanations.\n\n"
        f"### CONTEXT FROM DOCUMENTS:\n{context}"
    )

    # 3. Call Ollama with the context-aware prompt
    response = ollama.chat(
        model='llama2:7b',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': f"Translate: {text}"}
        ],
        options={'temperature': 0}
    )

    translation = response.message.content.strip()
    
    print(f"Original: {text}")
    print(f"English:  {translation}\n")
    my_rag_translations.append(translation)

HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Original: Buenos días.
English:  Good morning.



HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Original: Buenas tardes.
English:  Good afternoon.



HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


...

Original: ¿Tiene…?
English:  Here are the translations of "¿Tienes..." and its variations:

* ¿Tienes...? - Do you have...?
* ¿Tenéis...? - Do you have...? (plural)
* ¿Te gusta...? - Do you like...?
* ¿Le gusta...? - Does he like...?
* ¿Ustedes gustan...? - Do you like...? (plural)
* ¿Sabés por qué...? - Do you know why...?
* ¿Confías en mí? - Do you trust me?
* ¿Confía en mí? - Do you trust me?
* ¿Confiáis en mí? - Do you trust me?
* ¿Me quedo? - Should I stay?
* Cierra la puerta. - Shut the door.
* Cierra la puerta. - Shut the gate.
* Siéntate junto a mí. - Sit beside me.
* Sentate Tomás. - Sit down, Tom.
* Algunos peces vuelan. - Some fish fly.
* Alguien llegó. - Somebody came.
* Habla claramente. - Speak clearly.
* Hablen claramente. - Speak clearly.
* Hablad claramente. - Speak clearly.
* Habla quedito. - Speak quietly.
* Habla calladito. - Speak quietly.
* Párense en línea. - Stand in line.
* Empieza a correr. - Start running.
* Comienza a correr. - Start running.
* Empieza a can

HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Original: Me gustaría…
English:  Sure! Here are the translations for "Me gustaría..." in English:

* I would like...
* I would love...
* I would enjoy...
* I would appreciate...
* I would be happy to...

For example:

* Me gustaría ir al cine. (I would like to go to the movies.)
* Me gustaría probar esa nueva comida. (I would like to try that new food.)
* Me gustaría conocer a más personas. (I would like to meet more people.)
* Me gustaría aprender un nuevo idioma. (I would like to learn a new language.)
* Me gustaría tener un perro. (I would like to have a dog.)



HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Original: Quisiera…
English:  Would you like...?

* Quisiera un vaso: I want a glass.
* Quiero un cuchillo: I want a knife.
* Quiero un perrito: I want a puppy.
* Quiero un cachorrito: I want a puppy.
* Quiero otro: I want another.
* Quiero justicia: I want justice.
* Quiero palomitas: I want popcorn.
* Quiero palomitas de maíz: I want popcorn.
* Dije que te detuvieras: I said stop.
* Yo dije eso: I said that.
* Yo vi un OVNI: I saw a UFO.
* Vi a un perro: I saw a dog.
* Vi un perro: I saw a dog.
* Yo vi un perro: I saw a dog.
* Vendo autos: I sell cars.
* Vendo coches: I sell cars.
* Vendo pescado: I sell fish.
* Vendo jabón: I sell soap.
* Me debería ir: I should go.
* Huelo a gas: I smell gas.
* Huelo gas: I smell gas.
* Me huele a gas: I smell gas.
* Te lo agradezco: I thank you.
* Confío en Tom: I trust Tom.
* Yo confío en ella: I trust her.
* Yo confío en él: I trust him.
* Confío en ti: I trust you.
* Yo voté que sí: I voted yes.
* Quiero huevos: I want eggs.
* Quiero el mío: I 

In [22]:
bleu = sacrebleu.corpus_bleu(my_rag_translations, my_english_text)
chrf = sacrebleu.corpus_chrf(my_rag_translations, my_english_text)

print(f"BLEU Score: {bleu.score:.2f}")
print(f"chrF Score: {chrf.score:.2f}")

BLEU Score: 5.89
chrF Score: 35.71


In [23]:
# 1. Setup your data
# 'src' = original Spanish, 'mt' = your Ollama output, 'ref' = ground truth
data = [
    {
        "src": src,
        "mt": mt,
        "ref": ref
    }
    for src, mt, ref in zip(my_spanish_text, my_rag_translations, my_english_text)
]

print(f"CUDA Available: {torch.cuda.is_available()}")

# 2. Load the pre-trained model
# 'wmt22-comet-da' is a standard high-performing version

# 3. Calculate scores
model_output = model.predict(data, batch_size=8, gpus=1) # Set gpus=1 if you have a GPU

# 4. Access the results
print(f"System Score: {model_output.system_score}") 
# Individual scores are in model_output.scores

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


CUDA Available: True


/home/jovyan/.conda/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 14/14 [00:03<00:00,  3.56it/s]


System Score: 0.7104155059765886


In [24]:
my_QLoRA_RAG_translations = []

for text in my_spanish_text:
    # 1. Retrieve context relative to the Spanish snippet
    docs = retriever.invoke(text)
    context = "\n".join([doc.page_content for doc in docs])

    # 2. Integrate context into your specific QLoRA prompt template
    # We place the context BEFORE the translation task so the model "reads" it first
    prompt = (
        f"Context: {context}\n"
        f"Spanish: {text}\n"
        "English:"
    )

    # 3. Generate using your fine-tuned model
    response = ollama.generate(
        model='qlorallamatranslator',      
        prompt=prompt,
        options={
            'temperature': 0,
            'stop': ["Spanish:", "\n", "English:", "Context:"] # Added "Context:" as a stop sequence
        }
    )
    
    translation = response['response'].strip()

    print(f"Original: {text}")
    print(f"English:  {translation}\n")
    my_QLoRA_RAG_translations.append(translation)

HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


Original: Buenos días.
English:  Good morning.



HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


Original: Buenas tardes.
English:  Señora Presidenta, Señora Comisaria, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señora Comisaria, Señora Presidenta, Señ

HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


...

Original: ¿Tiene…?
English:  Señora Presidenta, eso es lo que nos ha dicho el Sr. Presidente del Consejo, Sr. Prodi, que nos ha dicho el Sr. Presidente del Consejo, Sr. Prodi, que nos ha dicho el Sr. Presidente del Consejo, Sr. Prodi, que nos ha dicho el Sr. Presidente del Consejo, Sr. Prodi, que nos ha dicho el Sr. Presidente del Consejo, Sr. Prodi, que nos ha dicho el Sr. Presidente del Consejo, Sr. Prodi, que nos ha dicho el Sr. Presidente del Consejo, Sr. Prodi, que nos ha dicho el Sr. Presidente del Consejo, Sr. Prodi, que nos ha dicho el Sr. Presidente del Consejo, Sr. Prodi, que nos ha dicho el Sr. Presidente del Consejo, Sr. Prodi, que nos ha dicho el Sr. Presidente del Consejo, Sr. Prodi, que nos ha dicho el Sr. Presidente del Consejo, Sr. Prodi, que nos ha dicho el Sr. Presidente del Consejo, Sr. Prodi, que nos ha dicho el Sr. Presidente del Consejo, Sr. Prodi, que nos ha dicho el Sr. Presidente del Consejo, Sr. Prodi, que nos ha dicho el Sr. Presidente del Consejo, Sr. Prodi

HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


Original: Me gustaría…
English:  I would like to…



HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


Original: Quisiera…
English:  I would like to…



In [25]:
bleu = sacrebleu.corpus_bleu(my_QLoRA_RAG_translations, my_english_text)
chrf = sacrebleu.corpus_chrf(my_QLoRA_RAG_translations, my_english_text)

print(f"BLEU Score: {bleu.score:.2f}")
print(f"chrF Score: {chrf.score:.2f}")

BLEU Score: 0.02
chrF Score: 0.26


In [26]:
# 1. Setup your data
# 'src' = original Spanish, 'mt' = your Ollama output, 'ref' = ground truth
data = [
    {
        "src": src,
        "mt": mt,
        "ref": ref
    }
    for src, mt, ref in zip(my_spanish_text, my_QLoRA_RAG_translations, my_english_text)
]

print(f"CUDA Available: {torch.cuda.is_available()}")

# 2. Load the pre-trained model
# 'wmt22-comet-da' is a standard high-performing version

# 3. Calculate scores
model_output = model.predict(data, batch_size=8, gpus=1) # Set gpus=1 if you have a GPU

# 4. Access the results
print(f"System Score: {model_output.system_score}") 
# Individual scores are in model_output.scores

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


CUDA Available: True


/home/jovyan/.conda/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 14/14 [00:02<00:00,  5.64it/s]


System Score: 0.7593914286957847


## Healthcare Translation Models

In [18]:
#Test dataset
from datasets import load_dataset
dataset = load_dataset("SINAI/ALIA-parallel-translation", split="train")

biomedical = dataset.filter(lambda x: x['id'].startswith('00'))

biomedical_test = biomedical[500001: 500151]

english_text = biomedical_test['text_en']
spanish_text = biomedical_test['text_es']

print(sum(text.count('.') for text in spanish_text))

Loading dataset shards:   0%|          | 0/35 [00:00<?, ?it/s]

1320


# Mistral

In [20]:
base_translations = []

for text in spanish_text:
    response = ollama.chat(
        model='mistral',
        messages=[
            # System role sets the behavior
            {'role': 'system', 'content': 'You are a professional translator. Translate the Spanish text to English. Provide only the translation without quotes or explanations.'},
            {'role': 'user', 'content': text}
        ],
        options={'temperature': 0}
    )

    #Print the results
    print(f"Original: {text}")
    print(f"English:  {response.message.content}\n")
    base_translations.append(response.message.content)

Original:  El hemangioma microvenular (HVM) es un tumor vascular benigno, poco frecuente, de crecimiento lento, que típicamente se presenta como una placa o nódulo solitario en el tronco o las extremidades de adultos de mediana edad. Una minoría de HVM presenta múltiples lesiones que se adquieren de forma gradual o repentina (HVM eruptiva). Presentamos el caso de una mujer de 53 años que desarrolló progresivamente numerosas HVM bilaterales que se presentaron como agrandamiento, bronceado, eritematosa a máculas violáceas y axilas proximales. Dos biopsias mostraron las venulas de ramificación irregular con lumina inconspicua sin atipia endotelial y asociada a fibrosis dérmica característica de la HMVM. Inmunofenotípicamente, el endotelio expresó tumor de Wilms 1, CD31, CD34 y proteína transportadora de glucosa tipo eritrocito (GLUT-1) GLUT-1 focalmente y fue negativo para el virus del herpes humano 8.40. Además, numerosas células fusiformes dérmicas que expresan CD34 y procolágenos, fibr

In [21]:
import sacrebleu
base_translations = [s.strip() for s in base_translations]

bleu = sacrebleu.corpus_bleu(base_translations, english_text)
chrf = sacrebleu.corpus_chrf(base_translations, english_text)

print(f"BLEU Score: {bleu.score:.2f}")
print(f"chrF Score: {chrf.score:.2f}")

BLEU Score: 0.03
chrF Score: 0.56


In [23]:
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "info"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
import lightning.pytorch as pl
import pytorch_lightning
from comet import download_model, load_from_checkpoint

# 1. The Bridge Fix
# Ensure the legacy mapping exists for COMET's internal calls
if not hasattr(pytorch_lightning, 'utilities'):
    pytorch_lightning.utilities = pl.utilities

# 2. Model Loading
# It is often safer to download and load in one go if you aren't managing paths manually
model_path = download_model("Unbabel/wmt22-comet-da")
model = load_from_checkpoint(model_path)

# 3. Data Preparation
# Ensure your input lists (my_spanish_text, etc.) are already defined
data = [
    {"src": src, "mt": mt, "ref": ref}
    for src, mt, ref in zip(spanish_text, base_translations, english_text)
]

# 4. Predict Logic
# Note: Newer versions of COMET use 'gpus' as an integer or list.
# If you are on a CPU-only machine, set gpus=0.
model_output = model.predict(
    data, 
    batch_size=32, 
    gpus=1 if torch.cuda.is_available() else 0
)

# 5. Output
print(f"System Score: {model_output.system_score:.4f}")

[pytorch_lightning.utilities.migration.utils|INFO]Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/home/jovyan/.conda/lib/python3.13/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
[pytorch_lightning.utilities.rank_zero|INFO]GPU available: True (cuda), used: True
[pytorch_lightning.utilities.rank_zero|INFO]TPU available: False, using: 0 TPU cores
[pytorch_lightning.utilities.rank_zero|INFO]💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning

System Score: 0.8352


In [3]:
#Implement QLoRA from Unsloth
import os
os.environ["TORCH_COMPILE_DISABLE"] = "1"

from unsloth import FastLanguageModel, unsloth_train
import torch
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorWithPadding
from datasets import load_dataset

torch.cuda.empty_cache()

# --- 1. Load Model & Tokenizer ---
max_seq_length = 512
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-bnb-4bit", # Pre-quantized for speed
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# --- 2. Add QLoRA Adapters ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank: higher = more "memory" but more VRAM
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0.1, # Optimized at 0
    bias = "none",    # Optimized at none
    use_gradient_checkpointing = "unsloth",
)

# --- 3. Prepare Dataset ---
prompt_style = "Spanish: {}\nEnglish: {}"

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" 

def tokenize_function(examples):
    # This maps perfectly to "text_es" and "text_en" from your sample instance
    texts = [
        f"Spanish: {es}\nEnglish: {en}{tokenizer.eos_token}" 
        for es, en in zip(examples["text_es"], examples["text_en"])
    ]
    
    tokenized = tokenizer(
        texts,
        truncation = True,
        max_length = max_seq_length,
        padding = "max_length",
    )
    
    tokenized["labels"] = [list(ids) for ids in tokenized["input_ids"]]
    return tokenized

# Load your dataset
dataset = load_dataset("SINAI/ALIA-parallel-translation", split="train")

biomedical = dataset.filter(lambda x: x['id'].startswith('00'))

# Grab a clean 5,000 example subset safely
biomedical = biomedical.take(50000)

# Process the data
processed_dataset = biomedical.map(
    tokenize_function,
    batched = True,
    remove_columns = ["id", "text_es", "text_en"], # This drops 'id', 'text_es', and 'text_en'
)

# --- 4. Start Training ---
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = processed_dataset,
    #ataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer),
    packing = True,
    args = TrainingArguments(
        per_device_train_batch_size = 1, # You can likely increase this with Unsloth
        gradient_accumulation_steps = 16,
        warmup_steps = 5,
        #max_steps = 60,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        group_by_length = False,
        report_to = 'none', 
        average_tokens_across_devices=False,
        remove_unused_columns = False,
    ),
)
unsloth_train(trainer)

# --- 5. After trainer.train(), run Inference ---
FastLanguageModel.for_inference(model) # Enable 2x faster inference

==((====))==  Unsloth 2026.5.10: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    Tesla V100-PCIE-32GB. Num GPUs = 4. Max memory: 31.739 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu126. CUDA: 7.0. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Loading dataset shards:   0%|          | 0/35 [00:00<?, ?it/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50,000 | Num Epochs = 1 | Total steps = 782
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 16 x 1) = 64
 "-____-"     Trainable parameters = 41,943,040 of 7,283,675,136 (0.58% trained)


Step,Training Loss
1,2.374700
2,2.418000
3,1.836300
4,1.552300
5,1.537400
6,1.504600
7,1.461700
8,1.474400
9,1.475800
10,1.499100


comet_ml is installed but the Comet API Key is not configured. Please set the `COMET_API_KEY` environment variable to enable Comet logging. Check out the documentation for other ways of configuring it: https://www.comet.com/docs/v2/guides/experiment-management/configure-sdk/#set-the-api-key
comet_ml is installed but the Comet API Key is not configured. Please set the `COMET_API_KEY` environment variable to enable Comet logging. Check out the documentation for other ways of configuring it: https://www.comet.com/docs/v2/guides/experiment-management/configure-sdk/#set-the-api-key


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
      

In [4]:
model.save_pretrained_gguf("QLoRAModelHealth", tokenizer, quantization_method = "q4_k_m")

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /home/jovyan/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00003.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

...

Unsloth: Preparing safetensor model files:  67%|██████▋   | 2/3 [01:16<00:38, 38.49s/it]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 3/3 [00:57<00:00, 19.16s/it]


Unsloth: Merge process complete. Saved to `/home/jovyan/work/Ollama Models/QLoRAModelHealth`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['QLoRAModelHealth_gguf/mistral-7b.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['QLoRAModelHealth_gguf/mistral-7b.Q4_K_M.gguf']
Unsloth: No Ollama template m

{'save_directory': 'QLoRAModelHealth',
 'gguf_directory': 'QLoRAModelHealth_gguf',
 'gguf_files': ['QLoRAModelHealth_gguf/mistral-7b.Q4_K_M.gguf'],
 'modelfile_location': None,
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [7]:
!ollama create qloratranslatorhealth -f "/home/jovyan/work/Ollama Models/QLoRAModelHealth_gguf/Modelfile"

]11;?\gathering model components ⠙ gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠧ gathering model components ⠏ gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gathering model components ⠴ gathering model components ⠴ gathering model components ⠧ gathering model components ⠇ gathering model components ⠏ gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠸ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠏ gathering model components ⠋ gathering model components ⠙ gathering model components ⠙ gathering model components ⠸ gathering model components ⠼ gather

In [24]:
QLoRA_translations = []

for text in spanish_text:
    prompt = f"Spanish: {text}\nEnglish:"
    response = ollama.generate(
        model='qloratranslatorhealth',      
        prompt=prompt,
        options={'temperature': 0,
                'stop': ["Spanish:", "\n", "English:"]}
    )
    translation = response['response'].strip()

    #Print the results
    print(f"Original: {text}")
    print(f"English:  {translation}\n")
    QLoRA_translations.append(translation)


Original:  El hemangioma microvenular (HVM) es un tumor vascular benigno, poco frecuente, de crecimiento lento, que típicamente se presenta como una placa o nódulo solitario en el tronco o las extremidades de adultos de mediana edad. Una minoría de HVM presenta múltiples lesiones que se adquieren de forma gradual o repentina (HVM eruptiva). Presentamos el caso de una mujer de 53 años que desarrolló progresivamente numerosas HVM bilaterales que se presentaron como agrandamiento, bronceado, eritematosa a máculas violáceas y axilas proximales. Dos biopsias mostraron las venulas de ramificación irregular con lumina inconspicua sin atipia endotelial y asociada a fibrosis dérmica característica de la HMVM. Inmunofenotípicamente, el endotelio expresó tumor de Wilms 1, CD31, CD34 y proteína transportadora de glucosa tipo eritrocito (GLUT-1) GLUT-1 focalmente y fue negativo para el virus del herpes humano 8.40. Además, numerosas células fusiformes dérmicas que expresan CD34 y procolágenos, fibr

In [25]:
QLoRA_translations = [s.strip() for s in QLoRA_translations]

bleu = sacrebleu.corpus_bleu(QLoRA_translations, english_text)
chrf = sacrebleu.corpus_chrf(QLoRA_translations, english_text)

print(f"BLEU Score: {bleu.score:.2f}")
print(f"chrF Score: {chrf.score:.2f}")

BLEU Score: 0.04
chrF Score: 0.58


In [26]:
data = [
    {
        "src": src,
        "mt": mt,
        "ref": ref
    }
    for src, mt, ref in zip(spanish_text, QLoRA_translations, english_text)
]


# 3. Calculate scores
model_output = model.predict(data, batch_size=8, gpus=1) # Set gpus=1 if you have a GPU

# 4. Access the results
print(f"System Score: {model_output.system_score}") 

[pytorch_lightning.utilities.rank_zero|INFO]GPU available: True (cuda), used: True
[pytorch_lightning.utilities.rank_zero|INFO]TPU available: False, using: 0 TPU cores
[pytorch_lightning.utilities.rank_zero|INFO]💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
[pytorch_lightning.utilities.rank_zero|INFO]💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
[pytorch_lightning.accelerators.cuda|INFO]LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]
/home/jovyan/.conda/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting

System Score: 0.8403091289599737


# Llama

In [27]:
base_translations = []

for text in spanish_text:
    response = ollama.chat(
        model='llama2:7b',
        messages=[
            # System role sets the behavior
            {'role': 'system', 'content': 'You are a professional translator. Translate the Spanish text to English. Provide only the translation without quotes or explanations.'},
            {'role': 'user', 'content': text}
        ],
        options={'temperature': 0}
    )

    #Print the results
    print(f"Original: {text}")
    print(f"English:  {response.message.content}\n")
    base_translations.append(response.message.content)

Original:  El hemangioma microvenular (HVM) es un tumor vascular benigno, poco frecuente, de crecimiento lento, que típicamente se presenta como una placa o nódulo solitario en el tronco o las extremidades de adultos de mediana edad. Una minoría de HVM presenta múltiples lesiones que se adquieren de forma gradual o repentina (HVM eruptiva). Presentamos el caso de una mujer de 53 años que desarrolló progresivamente numerosas HVM bilaterales que se presentaron como agrandamiento, bronceado, eritematosa a máculas violáceas y axilas proximales. Dos biopsias mostraron las venulas de ramificación irregular con lumina inconspicua sin atipia endotelial y asociada a fibrosis dérmica característica de la HMVM. Inmunofenotípicamente, el endotelio expresó tumor de Wilms 1, CD31, CD34 y proteína transportadora de glucosa tipo eritrocito (GLUT-1) GLUT-1 focalmente y fue negativo para el virus del herpes humano 8.40. Además, numerosas células fusiformes dérmicas que expresan CD34 y procolágenos, fibr

In [28]:
import sacrebleu
base_translations = [s.strip() for s in base_translations]

bleu = sacrebleu.corpus_bleu(base_translations, english_text)
chrf = sacrebleu.corpus_chrf(base_translations, english_text)

print(f"BLEU Score: {bleu.score:.2f}")
print(f"chrF Score: {chrf.score:.2f}")

BLEU Score: 0.04
chrF Score: 0.57


In [39]:
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "info"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
import lightning.pytorch as pl
import pytorch_lightning
from comet import download_model, load_from_checkpoint

# 1. The Bridge Fix
# Ensure the legacy mapping exists for COMET's internal calls
if not hasattr(pytorch_lightning, 'utilities'):
    pytorch_lightning.utilities = pl.utilities

# 2. Model Loading
# It is often safer to download and load in one go if you aren't managing paths manually
model_path = download_model("Unbabel/wmt22-comet-da")
model = load_from_checkpoint(model_path)

# 3. Data Preparation
# Ensure your input lists (my_spanish_text, etc.) are already defined
data = [
    {"src": src, "mt": mt, "ref": ref}
    for src, mt, ref in zip(spanish_text, base_translations, english_text)
]

# 4. Predict Logic
# Note: Newer versions of COMET use 'gpus' as an integer or list.
# If you are on a CPU-only machine, set gpus=0.
model_output = model.predict(
    data, 
    batch_size=32, 
    gpus=1 if torch.cuda.is_available() else 0
)

# 5. Output
print(f"System Score: {model_output.system_score:.4f}")

[pytorch_lightning.utilities.migration.utils|INFO]Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/home/jovyan/.conda/lib/python3.13/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
[pytorch_lightning.utilities.rank_zero|INFO]GPU available: True (cuda), used: True
[pytorch_lightning.utilities.rank_zero|INFO]TPU available: False, using: 0 TPU cores
[pytorch_lightning.utilities.rank_zero|INFO]💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning

System Score: 0.8273


In [33]:
import os
os.environ["TORCH_COMPILE_DISABLE"] = "1"

from unsloth import FastLanguageModel, unsloth_train
import torch
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorWithPadding
from datasets import load_dataset


torch.cuda.empty_cache()

# --- 1. Load Model & Tokenizer ---
max_seq_length = 512
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-2-7b-chat-bnb-4bit", # Pre-quantized for speed
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# --- 2. Add QLoRA Adapters ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank: higher = more "memory" but more VRAM
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0.1, # Optimized at 0
    bias = "none",    # Optimized at none
    use_gradient_checkpointing = "unsloth",
)


# --- 3. Prepare Dataset ---
prompt_style = "Spanish: {}\nEnglish: {}"

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" 

def tokenize_function(examples):
    # This maps perfectly to "text_es" and "text_en" from your sample instance
    texts = [
        f"Spanish: {es}\nEnglish: {en}{tokenizer.eos_token}" 
        for es, en in zip(examples["text_es"], examples["text_en"])
    ]
    
    tokenized = tokenizer(
        texts,
        truncation = True,
        max_length = max_seq_length,
        padding = "max_length",
    )
    
    tokenized["labels"] = [list(ids) for ids in tokenized["input_ids"]]
    return tokenized

# Load your dataset
dataset = load_dataset("SINAI/ALIA-parallel-translation", split="train")

biomedical = dataset.filter(lambda x: x['id'].startswith('00'))

# Grab a clean 5,000 example subset safely
biomedical = biomedical.take(50000)

# Process the data
processed_dataset = biomedical.map(
    tokenize_function,
    batched = True,
    remove_columns = ["id", "text_es", "text_en"], # This drops 'id', 'text_es', and 'text_en'
)

# --- 4. Start Training ---
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = processed_dataset,
    #ataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer),
    packing = True,
    args = TrainingArguments(
        per_device_train_batch_size = 1, # You can likely increase this with Unsloth
        gradient_accumulation_steps = 16,
        warmup_steps = 5,
        #max_steps = 60,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        group_by_length = False,
        report_to = 'none', 
        average_tokens_across_devices=False,
        remove_unused_columns = False,
    ),
)
unsloth_train(trainer)

# --- 5. After trainer.train(), run Inference ---
FastLanguageModel.for_inference(model) # Enable 2x faster inference

==((====))==  Unsloth 2026.5.10: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla V100-PCIE-32GB. Num GPUs = 4. Max memory: 31.739 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu126. CUDA: 7.0. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading dataset shards:   0%|          | 0/35 [00:00<?, ?it/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50,000 | Num Epochs = 1 | Total steps = 782
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 16 x 1) = 64
 "-____-"     Trainable parameters = 39,976,960 of 6,778,392,576 (0.59% trained)


Step,Training Loss
1,2.761700
2,2.743700
3,2.682500
4,2.484900
5,2.258000
6,2.168500
7,2.028400
8,1.995600
9,1.916400
10,1.922700


comet_ml is installed but the Comet API Key is not configured. Please set the `COMET_API_KEY` environment variable to enable Comet logging. Check out the documentation for other ways of configuring it: https://www.comet.com/docs/v2/guides/experiment-management/configure-sdk/#set-the-api-key
comet_ml is installed but the Comet API Key is not configured. Please set the `COMET_API_KEY` environment variable to enable Comet logging. Check out the documentation for other ways of configuring it: https://www.comet.com/docs/v2/guides/experiment-management/configure-sdk/#set-the-api-key


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              

In [21]:
model.save_pretrained("QLoRALLamaModelHealth_Backup", tokenizer)

In [34]:
model.save_pretrained_gguf("QLoRALLamaModelHealth", tokenizer, quantization_method = "q4_k_m")

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /home/jovyan/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00003.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 3/3 [00:35<00:00, 11.79s/it]


Unsloth: Merge process complete. Saved to `/home/jovyan/work/Ollama Models/QLoRALLamaModelHealth`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['QLoRALLamaModelHealth_gguf/llama-2-7b-chat.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: All GGUF conversions completed successfully!
Generated files: ['QLoRALLamaModelHealth_gguf/llama-2-7b-chat.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /home/jovyan/.unsloth/llama.cpp/llama-cli --model QLoRALLamaModelHealth_gguf/llama-2-7b-chat.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to QLoRALLamaModelHealth_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f QLoRALLamaModelHealth_gguf/Modelfile


{'save_directory': 'QLoRALLamaModelHealth',
 'gguf_directory': 'QLoRALLamaModelHealth_gguf',
 'gguf_files': ['QLoRALLamaModelHealth_gguf/llama-2-7b-chat.Q4_K_M.gguf'],
 'modelfile_location': 'QLoRALLamaModelHealth_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': True}

In [35]:
!ollama create qlorallamatranslatorhealth -f "/home/jovyan/work/Ollama Models/QLoRALLamaModelHealth_gguf/Modelfile"

]11;?\gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠏ gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gathering model components ⠴ gathering model components ⠴ gathering model components ⠧ gathering model components ⠇ gathering model components ⠇ gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠏ gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gather

In [36]:
QLoRA_translations = []

for text in spanish_text:
    prompt = f"Spanish: {text}\nEnglish:"
    response = ollama.generate(
        model='qlorallamatranslatorhealth',      
        prompt=prompt,
        options={'temperature': 0,
                'stop': ["Spanish:", "\n", "English:"]}
    )
    translation = response['response'].strip()

    #Print the results
    print(f"Original: {text}")
    print(f"English:  {translation}\n")
    QLoRA_translations.append(translation)


Original:  El hemangioma microvenular (HVM) es un tumor vascular benigno, poco frecuente, de crecimiento lento, que típicamente se presenta como una placa o nódulo solitario en el tronco o las extremidades de adultos de mediana edad. Una minoría de HVM presenta múltiples lesiones que se adquieren de forma gradual o repentina (HVM eruptiva). Presentamos el caso de una mujer de 53 años que desarrolló progresivamente numerosas HVM bilaterales que se presentaron como agrandamiento, bronceado, eritematosa a máculas violáceas y axilas proximales. Dos biopsias mostraron las venulas de ramificación irregular con lumina inconspicua sin atipia endotelial y asociada a fibrosis dérmica característica de la HMVM. Inmunofenotípicamente, el endotelio expresó tumor de Wilms 1, CD31, CD34 y proteína transportadora de glucosa tipo eritrocito (GLUT-1) GLUT-1 focalmente y fue negativo para el virus del herpes humano 8.40. Además, numerosas células fusiformes dérmicas que expresan CD34 y procolágenos, fibr

In [37]:
QLoRA_translations = [s.strip() for s in QLoRA_translations]

bleu = sacrebleu.corpus_bleu(QLoRA_translations, english_text)
chrf = sacrebleu.corpus_chrf(QLoRA_translations, english_text)

print(f"BLEU Score: {bleu.score:.2f}")
print(f"chrF Score: {chrf.score:.2f}")

BLEU Score: 0.00
chrF Score: 3.91


In [40]:
data = [
    {
        "src": src,
        "mt": mt,
        "ref": ref
    }
    for src, mt, ref in zip(spanish_text, QLoRA_translations, english_text)
]


# 3. Calculate scores
model_output = model.predict(data, batch_size=8, gpus=1) # Set gpus=1 if you have a GPU

# 4. Access the results
print(f"System Score: {model_output.system_score}") 

[pytorch_lightning.utilities.rank_zero|INFO]GPU available: True (cuda), used: True
[pytorch_lightning.utilities.rank_zero|INFO]TPU available: False, using: 0 TPU cores
[pytorch_lightning.utilities.rank_zero|INFO]💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
[pytorch_lightning.utilities.rank_zero|INFO]💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
[pytorch_lightning.accelerators.cuda|INFO]LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]
/home/jovyan/.conda/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting

System Score: 0.14156972467899323
